# NB02 - Adam & AdamW
## Background
Adam (Kingma & Ba, 2014) unified momentum with per-parameter adaptive learning rates, combining RMSprop's second-moment estimate with a first-moment estimate. AdamW (Loshchilov & Hutter, 2019) decouples weight decay from the gradient update, restoring intended regularization behavior. This distinction matters enormously at scale: L2-regularized Adam weakens regularization because the weight penalty is scaled by the adaptive learning rate.

## What You'll Learn
- Adam from scratch: first moment, second moment, bias correction
- AdamW: decoupled weight decay applied directly to parameters
- Weight decay ablation on polynomial regression overfitting
- Bias correction: why it is critical at step 1

## Why This Matters
AdamW + cosine schedule is the de facto standard for training transformers. Understanding the weight decay decoupling explains why L2-regularized Adam underperforms AdamW on large models.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Adam:
    lr: float = 1e-3
    beta1: float = 0.9
    beta2: float = 0.999
    eps: float = 1e-8
    weight_decay: float = 0.0
    _m: Optional[np.ndarray] = field(default=None, init=False)
    _v: Optional[np.ndarray] = field(default=None, init=False)
    _t: int = field(default=0, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self._m is None:
            self._m = np.zeros_like(params)
            self._v = np.zeros_like(params)
        self._t += 1
        if self.weight_decay > 0:
            grads = grads + self.weight_decay * params  # L2 in gradient
        self._m = self.beta1 * self._m + (1 - self.beta1) * grads
        self._v = self.beta2 * self._v + (1 - self.beta2) * grads**2
        m_hat = self._m / (1 - self.beta1**self._t)
        v_hat = self._v / (1 - self.beta2**self._t)
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def reset(self):
        self._m = self._v = None
        self._t = 0

@dataclass
class AdamW:
    lr: float = 1e-3
    beta1: float = 0.9
    beta2: float = 0.999
    eps: float = 1e-8
    weight_decay: float = 0.01
    _m: Optional[np.ndarray] = field(default=None, init=False)
    _v: Optional[np.ndarray] = field(default=None, init=False)
    _t: int = field(default=0, init=False)

    def step(self, params: np.ndarray, grads: np.ndarray) -> np.ndarray:
        if self._m is None:
            self._m = np.zeros_like(params)
            self._v = np.zeros_like(params)
        self._t += 1
        self._m = self.beta1 * self._m + (1 - self.beta1) * grads
        self._v = self.beta2 * self._v + (1 - self.beta2) * grads**2
        m_hat = self._m / (1 - self.beta1**self._t)
        v_hat = self._v / (1 - self.beta2**self._t)
        update = self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
        return params * (1 - self.lr * self.weight_decay) - update  # decoupled

    def reset(self):
        self._m = self._v = None
        self._t = 0

print('Adam and AdamW implemented.')
print('Key difference:')
print('  Adam L2:  grads += wd * params  => wd scaled by 1/sqrt(v_hat)')
print('  AdamW:    params *= (1 - lr*wd) => wd independent of gradient scale')

In [ ]:
np.random.seed(42)
n_train, n_test = 20, 200
x_train = np.linspace(-1, 1, n_train)
x_test  = np.linspace(-1, 1, n_test)
true_fn = lambda x: np.sin(2 * np.pi * x)
y_train = true_fn(x_train) + 0.3 * np.random.randn(n_train)
y_test  = true_fn(x_test)

def poly_features(x, degree=9):
    return np.stack([x**i for i in range(degree+1)], axis=-1)

X_train = poly_features(x_train)
X_test  = poly_features(x_test)

def mse_and_grad(w, X, y):
    pred = X @ w
    err = pred - y
    loss = np.mean(err**2)
    grad = 2 * X.T @ err / len(y)
    return loss, grad

def train_optimizer(opt, X, y, n_steps=2000):
    w = np.zeros(X.shape[1])
    opt.reset()
    for _ in range(n_steps):
        loss, grad = mse_and_grad(w, X, y)
        w = opt.step(w, grad)
    return w

wd_values = [0.0, 0.001, 0.01, 0.1]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

test_losses = []
for wd in wd_values:
    opt = AdamW(lr=1e-2, weight_decay=wd)
    w = train_optimizer(opt, X_train, y_train)
    test_pred = X_test @ w
    test_loss = np.mean((test_pred - y_test)**2)
    test_losses.append(test_loss)
    axes[0].plot(test_pred, label=f'wd={wd} (MSE={test_loss:.3f})', alpha=0.8)

axes[0].plot(y_test, 'k--', linewidth=2, label='True function')
axes[0].set_title('AdamW Weight Decay Ablation')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].bar([str(wd) for wd in wd_values], test_losses, color='steelblue', alpha=0.8)
axes[1].set_title('Test MSE vs Weight Decay')
axes[1].set_xlabel('weight_decay'); axes[1].set_ylabel('Test MSE')
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(f'{BASE}/s30_02_weight_decay.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Optimal weight decay: {wd_values[np.argmin(test_losses)]}')

In [ ]:
print('=== Adam Bias Correction at Step 1 ===')
g = 1.0
b1, b2 = 0.9, 0.999
m, v = 0.0, 0.0
print(f'{'Step':>5} | {'m_raw':>8} | {'v_raw':>10} | {'m_hat':>8} | {'v_hat':>10} | {'update':>8}')
print('-' * 60)
for t in range(1, 11):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g**2
    m_hat = m / (1 - b1**t)
    v_hat = v / (1 - b2**t)
    update = m_hat / (np.sqrt(v_hat) + 1e-8)
    print(f'{t:>5} | {m:>8.4f} | {v:>10.6f} | {m_hat:>8.4f} | {v_hat:>10.6f} | {update:>8.4f}')
print()
print('Without bias correction at t=1: m/sqrt(v) ~ 3.16 (too large!)')
print('With bias correction at t=1:    m_hat/sqrt(v_hat) ~ 1.0 (correct)')